In [2]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
import os

load_dotenv()

os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY')
llm = ChatOpenAI(model="gpt-4.1-nano", temperature=0)

In [3]:
# Wikipedia Tool
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

api_wrapper = WikipediaAPIWrapper(top_k_results=1, doc_content_chars_max=200)
wikipedia = WikipediaQueryRun(api_wrapper=api_wrapper)

wikipedia.run("What is ChatGPT?")

'Page: ChatGPT\nSummary: ChatGPT is a generative artificial intelligence chatbot developed by OpenAI and released on November 30, 2022. It uses generative pre-trained transformers (GPTs), such as GPT-4o'

In [4]:
# Arxiv Tool
from langchain_community.tools import ArxivQueryRun
from langchain_community.utilities import ArxivAPIWrapper

arxiv_api_wrapper = ArxivAPIWrapper(top_k_results=1, doc_content_chars_max=1000)

arxiv = ArxivQueryRun(api_wrapper=arxiv_api_wrapper)

arxiv.run("391366950 Natural Language Processing: A Comprehensive Survey")

'Published: 2023-05-13\nTitle: Natural Language Reasoning, A Survey\nAuthors: Fei Yu, Hongbo Zhang, Prayag Tiwari, Benyou Wang\nSummary: This survey paper proposes a clearer view of natural language reasoning in\nthe field of Natural Language Processing (NLP), both conceptually and\npractically. Conceptually, we provide a distinct definition for natural\nlanguage reasoning in NLP, based on both philosophy and NLP scenarios, discuss\nwhat types of tasks require reasoning, and introduce a taxonomy of reasoning.\nPractically, we conduct a comprehensive literature review on natural language\nreasoning in NLP, mainly covering classical logical reasoning, natural language\ninference, multi-hop question answering, and commonsense reasoning. The paper\nalso identifies and views backward reasoning, a powerful paradigm for\nmulti-step reasoning, and introduces defeasible reasoning as one of the most\nimportant future directions in natural language reasoning research. We focus on\nsingle-modality

In [5]:
# Create Custom Tool to Load Webpage
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import OpenAIEmbeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter

# load webpage document
loader = WebBaseLoader("https://dzonetechnologies.com/")

docs = loader.load()

# Split text
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200).split_documents(docs)

vectordb = Chroma.from_documents(text_splitter, OpenAIEmbeddings())

# create retriever
retriever = vectordb.as_retriever()

retriever

USER_AGENT environment variable not set, consider setting it to identify your requests.


C:\Users\Muhammad Adeel\AppData\Local\Temp\ipykernel_23308\1757338408.py:15: LangChainDeprecationWarning: The class `OpenAIEmbeddings` was deprecated in LangChain 0.0.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import OpenAIEmbeddings``.
  vectordb = Chroma.from_documents(text_splitter, OpenAIEmbeddings())


VectorStoreRetriever(tags=['Chroma', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x000001C2ACD1BCB0>, search_kwargs={})

In [6]:
# Convert retriever into langchain custom tool
from langchain.tools.retriever import create_retriever_tool

dzone_retriever_tool = create_retriever_tool(
    retriever=retriever,
    name="dzone_technologies_retriever",
    description="Useful for answering questions about the services, solutions, and offerings provided by DZone Technologies. Retrieves information from the official DZone Technologies website, including pages related to cybersecurity, consulting, and IT solutions."
)

In [7]:
tools = [wikipedia, arxiv, dzone_retriever_tool]

In [10]:
# Agents

from langchain import hub
from langchain.agents import create_openai_tools_agent, AgentExecutor

prompt = hub.pull("hwchase17/openai-tools-agent")

agent = create_openai_tools_agent(llm=llm, tools=tools, prompt=prompt)

agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

# agent_executor.invoke({'input': "Which services Dzone Technologies is offering?"})
agent_executor.invoke({'input': "Artifical intelligence research paper"})










> Entering new AgentExecutor chain...

Invoking: `arxiv` with `{'query': 'Artificial intelligence'}`


Published: 2023-04-06
Title: The Governance of Physical Artificial Intelligence
Authors: Yingbo Li, Anamaria-Beatrice Spulber, Yucong Duan
Summary: Physical artificial intelligence can prove to be one of the most important
challenges of the artificial intelligence. The governance of physical
artificial intelligence would define its responsible intelligent application in
the society.I found a recent research paper titled "The Governance of Physical Artificial Intelligence" by Yingbo Li, Anamaria-Beatrice Spulber, and Yucong Duan, published on April 6, 2023. The paper discusses the challenges and governance of physical artificial intelligence and its responsible application in society. Would you like a more detailed summary or information on a specific aspect of this paper?

> Finished chain.


{'input': 'Artifical intelligence research paper',
 'output': 'I found a recent research paper titled "The Governance of Physical Artificial Intelligence" by Yingbo Li, Anamaria-Beatrice Spulber, and Yucong Duan, published on April 6, 2023. The paper discusses the challenges and governance of physical artificial intelligence and its responsible application in society. Would you like a more detailed summary or information on a specific aspect of this paper?'}